# Extract genes based on PCA

## PCA

In [1]:
import pandas as pd
%run "../Model/DataHelpers.ipynb"

In [2]:
print(f'Loading gene data - Start')
df = pd.read_csv('../Data/geneDataPreProcessed.csv')
print(f'Loading gene data - End')

Loading gene data - Start
Loading gene data - End


### Dataset split: training and test data

In [3]:
X, y, X_train, X_test, y_train, y_test, test_case_ids = split_data(df, "tnbc", True)

X_train.shape=(781, 19938)
X_test.shape=(196, 19938)
y_train.shape=(781,)
y_test.shape=(196,)


# Apply PCA

In [4]:
variant = FeatureVariant.AUTOMATED
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# Scale features (fit ONLY on train)
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# Apply PCA (fit ONLY on train)
pca = PCA(n_components=0.95, random_state=42)

X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca  = pca.transform(X_test_scaled)

print(f"Original features: {X_train.shape[1]}")
print(f"PCA components retained: {X_train_pca.shape[1]}")

# Build PCA DataFrames
pca_columns = [f'PC{i+1}' for i in range(X_train_pca.shape[1])]

df_train_pca = pd.DataFrame(X_train_pca, columns=pca_columns)
df_train_pca['tnbc'] = y_train.values

df_test_pca = pd.DataFrame(X_test_pca, columns=pca_columns)
df_test_pca['tnbc'] = y_test.values
df_test_pca['case_id'] = test_case_ids.values

Original features: 19938
PCA components retained: 535


In [5]:
print(df_train_pca.head())

         PC1        PC2        PC3        PC4        PC5        PC6  \
0  84.249000  20.278609  56.141925 -15.894505  34.147673  -5.375305   
1   4.335001  -3.003506  -6.123479 -33.500740 -45.895265 -16.805220   
2 -18.880805  26.391024   8.881719  28.301690 -35.667703  29.419789   
3  36.184210  38.842142  53.436617 -55.309951  17.070123  13.876680   
4  91.889723 -11.447379 -14.320143 -44.463471  25.030023  32.499043   

         PC7        PC8       PC9       PC10  ...     PC527     PC528  \
0  11.158841  -8.916769 -1.035906  -1.171060  ...  1.701484 -2.686439   
1   9.402011  -6.599400  1.570580  -6.018577  ... -0.405714 -1.086954   
2   4.177099 -28.646460 -8.768045   1.747355  ...  5.673668 -2.788206   
3   4.568879 -13.711150  3.185127  15.205936  ...  1.182778 -0.133396   
4   1.797694  14.134186  0.251899  -6.048918  ...  1.952831  1.385432   

      PC529     PC530     PC531     PC532     PC533     PC534     PC535   tnbc  
0 -0.436694  0.185119  2.520774  0.820054 -0.215621 -

In [6]:
print(df_test_pca.head())

         PC1        PC2        PC3        PC4        PC5        PC6  \
0 -14.670720 -15.851997   5.644188  30.291086 -14.651378 -22.046374   
1 -27.280529 -10.446520 -25.768137 -31.797642 -16.208498  -7.167662   
2 -35.343576   9.252483  28.649058 -10.476034 -35.329485  -7.970957   
3  31.968590   5.964668  38.248365 -61.454102  56.886382 -40.832785   
4  22.724470 -18.889962 -31.898267  -7.809805   3.879947  -1.727140   

         PC7        PC8        PC9       PC10  ...     PC528     PC529  \
0  13.575124   4.941383  -4.723264   5.112648  ...  0.520138 -0.218313   
1 -12.902581   0.133777 -34.432965  -2.597172  ... -0.373309  0.052620   
2   0.094341  -0.172495  -0.643546 -28.711332  ... -0.064139 -0.985373   
3 -26.859644   7.131318 -23.752246  -7.465803  ...  0.030182 -0.044106   
4  29.198530 -11.066581  28.821870  -2.412926  ... -1.331501  1.723969   

      PC530     PC531     PC532     PC533     PC534     PC535   tnbc  \
0  0.202137 -0.456800 -1.948747 -0.779733 -2.529080 -0.7

# Send dataframe to a csv

In [7]:
# Write PCA features to disk
# Separate train and test for use in models
df_train_pca.to_csv(f'../Data/interim/patient_genes_{variant}_train.csv', index=False)
df_test_pca.to_csv(f'../Data/interim/patient_genes_{variant}_test.csv', index=False)